In [0]:
# load libraries
from pyspark.sql.functions import col, when, sum, when, lit, coalesce, year
from pyspark.sql.window import Window
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score,f1_score


### Data Prepareation and Research Question Set Up
I import and use the dataset I created from previous homework2. The dataset is a race-level driver results， combining race results, driver information, and historical performance. Each row represents one driver in one race. It includes race outcome variables such as finishing position and points, driver characteristics such as name and age, and historical performance features. 

The research question would be: Can we predict whether a driver will outperform their starting grid position?

So, for each driver-race row, we will create a new binary variable. `beat_grid = 1` if the driver finished better than they start position, `beat_grid = 0` otherwise. In F1, a smaller finishing position number is better. For example, if we have start 8th, finish 5th, the driver improved, so `beat_grid = 1`. If the driver started 3rd but finish 7th, it's worse and `beat_grid = 0`.

In addition, “outperforming the starting grid” is defined as finishing in a strictly better position than the starting grid position. Therefore, `beat_grid=1` when `positionOrder < grid`, and 0 otherwise. Drivers who finished in the same position as they started are not considered to have outperformed their grid position.

In [0]:
# Load the saved results_age dataset from last HW
df_results_age = spark.read.csv("/Volumes/gr5069/sz3415/assignments/f1_results_age.csv",header=True)
display(df_results_age)

In [0]:
# Create a copy of the dataset for modeling
df_model = df_results_age

# Define the target variable:
# beat_grid = 1 if the driver finished in a better position than they started, 0 otherwise
df_model = df_model.withColumn(
    "beat_grid",
    when(col("positionOrder") < col("grid"), 1).otherwise(0)) #Smaller number means better position

# Confirm the target was created correctly
display(
    df_model.select("raceId", "driverId", "grid", "positionOrder", "beat_grid"))

#### Choose predictor variables
We want to use information that would have been known before the race, so should avoid variables that directly reveal the race outcome.

We also need to rebuild first_so_far, seconds_so_far, and thirds_so_far to avoid leakage, because they include current race now. Then, the model is accidentally using information from the outcome it is supposed to predict. To make predictions, we want to only inlcude the results from previous races. 

In [0]:
# Create indicator columns for current race finishing positions
df_model = df_model.withColumn(
    "first_place",
    when(col("positionOrder") == 1, 1).otherwise(0)
).withColumn(
    "second_place",
    when(col("positionOrder") == 2, 1).otherwise(0)
).withColumn(
    "third_place",
    when(col("positionOrder") == 3, 1).otherwise(0)
    )

# Define a driver-level historical window
# It includes all prior races, but not the current race
history_window = Window.partitionBy("driverId") \
    .orderBy("race_date", "raceId") \
    .rowsBetween(Window.unboundedPreceding, -1)

# Build cumulative counts(1st,2nd,3rd places) for previous races
df_model = df_model.withColumn(
    "1st_before_race",
    coalesce(sum("first_place").over(history_window), lit(0))
).withColumn(
    "2nd_before_race",
    coalesce(sum("second_place").over(history_window), lit(0))
).withColumn(
    "3rd_before_race",
    coalesce(sum("third_place").over(history_window), lit(0))
    )

# Create total prior podium count
df_model = df_model.withColumn(
    "podiums_before_race",
    col("1st_before_race") + col("2nd_before_race") + col("3rd_before_race"))


In [0]:
# Select the columns to keep for modeling
# include basic race/driver features plus historical performance columns
df_model = df_model.select(
    "raceId",
    "driverId",
    "race_date",
    "grid",
    "Age",
    "1st_before_race",
    "2nd_before_race",
    "3rd_before_race",
    "podiums_before_race",
    "beat_grid"
    )


# Preview the cleaned modeling columns
display(df_model.orderBy("driverId", "race_date"))

In [0]:
# Double check  missing values in main columns
missing_summary = df_model.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ["grid", "Age", "1st_before_race", "2nd_before_race", "3rd_before_race", "beat_grid"]
    ])

display(missing_summary)

In [0]:
# Import drivers dataset
df_drivers = spark.read.csv('/Volumes/gr5069/raw/f1_data/drivers.csv',header=True)

# Join driver names into the modeling dataset for future interpretation
df_model = df_model.join(
    df_drivers.select("driverId", "forename", "surname"),
    on="driverId",
    how="left"
    )

display(df_model)

#### Build and train the model
For this racing data, we will split the data by time instead of randomly. So, we train on earlier races and test on later races. If we randomly split the data, the model may learn from future races and then be tested on earlier races, which is unrealistic. 


In [0]:
# Create a race year column from race_date
df_model = df_model.withColumn("race_year", year(col("race_date"))) # extracts year from the race date

# Inspect year counts
display(df_model.groupBy("race_year").count().orderBy("race_year"))

To keep the split close to the common 80/20 rule, we will use the earlier approximately 80% of observations as the training set and the later approximately 20% as the test set. Based on the year-count distribution, 2012 is a reasonable cutoff point.

In [0]:
# Split data by time
# Train on earlier races, test on later races
train_df = df_model.filter(col("race_year") < 2012)
test_df = df_model.filter(col("race_year") >= 2012)

# Check the size of each split
print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

In [0]:
# Select the modeling columns use
feature_cols = [
    "grid",
    "Age",
    "1st_before_race",
    "2nd_before_race",
    "3rd_before_race",
    "podiums_before_race"]

# Set target for ML model
target_col = "beat_grid"

# Convert Spark DataFrames to pandas DataFrames for scikit-learn
train_pd = train_df.select(feature_cols + [target_col]).toPandas()
test_pd = test_df.select(feature_cols + [target_col]).toPandas()

# Create X and y for training and testing
X_train = train_pd[feature_cols]
X_test = test_pd[feature_cols]
y_train = train_pd[target_col]
y_test = test_pd[target_col]

# Check the shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

#### Train Logistic Regression and log with MLflow

In [0]:
def log_rf(experimentID, run_name, params, X_train, X_test, y_train, y_test, feature_cols):
  import os
  import pandas as pd
  import matplotlib.pyplot as plt
  import mlflow.sklearn
  import seaborn as sns
  import tempfile
  
  from sklearn.ensemble import RandomForestClassifier
  from sklearn.metrics import (
      accuracy_score,
      precision_score,
      recall_score,
      f1_score,
      roc_auc_score,
      confusion_matrix,
      ConfusionMatrixDisplay,
      RocCurveDisplay
      )

  with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
    # Create model, train it, and create predictions
    rf = RandomForestClassifier(**params)
    rf.fit(X_train, y_train)
    predictions = rf.predict(X_test)
    probabilities = rf.predict_proba(X_test)[:, 1]

    # Log model
    mlflow.sklearn.log_model(rf, "random-forest-model")

    # Log params
    [mlflow.log_param(param, value) for param, value in params.items()]

    # Create metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, probabilities)

    print("  accuracy: {}".format(accuracy))
    print("  precision: {}".format(precision))
    print("  recall: {}".format(recall))
    print("  f1: {}".format(f1))
    print("  roc_auc: {}".format(roc_auc))

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", roc_auc)

    # Create feature importance
    importance = pd.DataFrame(
        list(zip(feature_cols, rf.feature_importances_)),
        columns=["Feature", "Importance"]
        ).sort_values("Importance", ascending=False)

    # Log importances using temporary file
    temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
    temp_name = temp.name
    try:
      importance.to_csv(temp_name, index=False)
      mlflow.log_artifact(temp_name, "feature_importance.csv")
    finally:
      temp.close()  # Delete the temp file

    # Create confusion matrix plot
    fig, ax = plt.subplots()
    cm = confusion_matrix(y_test, predictions)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")

    # Log confusion matrix using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="confusion-matrix-", suffix=".png")
    temp_name = temp.name
    try:
      fig.savefig(temp_name)
      mlflow.log_artifact(temp_name, "confusion_matrix.png")
    finally:
      temp.close()  # Delete the temp file

    display(fig)
    plt.close(fig)

    # Create predictions artifact
    predictions_df = pd.DataFrame({
        "actual": y_test,
        "predicted": predictions,
        "predicted_probability": probabilities
    })

    temp = tempfile.NamedTemporaryFile(prefix="predictions-", suffix=".csv")
    temp_name = temp.name
    try:
      predictions_df.to_csv(temp_name, index=False)
      mlflow.log_artifact(temp_name, "predictions.csv")
    finally:
      temp.close()  # Delete the temp file

    return run.info.run_id

run with different parameters

In [0]:
params = {
  "n_estimators": 50,
  "max_depth": 5,
  "random_state": 42}

log_rf(experimentID, "Run 1: depth5 trees50", params, X_train, X_test, y_train, y_test, feature_cols)

In [0]:
params = {
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": 42}

log_rf(experimentID, "Run 2：RF depth5 trees100", params, X_train, X_test, y_train, y_test, feature_cols)

In [0]:
params = {
    "n_estimators": 150,
    "max_depth": 5,
    "random_state": 42}

log_rf(experimentID, "Run 3： RF depth5 trees150", params, X_train, X_test, y_train, y_test, feature_cols)

In [0]:
params = {
    "n_estimators": 50,
    "max_depth": 10,
    "random_state": 42}

log_rf(experimentID, "Run 4： RF depth10 trees50", params, X_train, X_test, y_train, y_test, feature_cols)

In [0]:
params = {
    "n_estimators": 100,
    "max_depth": 10,
    "random_state": 42}

log_rf(experimentID, "Run 5： RF depth10 trees100", params, X_train, X_test, y_train, y_test, feature_cols)

In [0]:
params = {
    "n_estimators": 150,
    "max_depth": 10,
    "random_state": 42}

log_rf(experimentID, "Run 6： RF depth10 trees150", params, X_train, X_test, y_train, y_test, feature_cols)

In [0]:
params = {
    "n_estimators": 50,
    "max_depth": 15,
    "random_state": 42}

log_rf(experimentID, "Run 7: RF depth15 trees50", params, X_train, X_test, y_train, y_test, feature_cols)

In [0]:
params = {
    "n_estimators": 100,
    "max_depth": 15,
    "random_state": 42}

log_rf(experimentID, "Run 8: RF depth15 trees100", params, X_train, X_test, y_train, y_test, feature_cols)

In [0]:
params = {
    "n_estimators": 150,
    "max_depth": 15,
    "random_state": 42}

log_rf(experimentID, "Run 9: RF depth15 trees150", params, X_train, X_test, y_train, y_test, feature_cols)

In [0]:
params = {
    "n_estimators": 200,
    "max_depth": 20,
    "random_state": 42}

log_rf(experimentID, "Run 10: RF depth20 trees200", params, X_train, X_test, y_train, y_test, feature_cols)

#### Best model
Among all 10 RF runs, Model 6 is selected as the best model. It has the  highest ROC-AUC (0.762), the highest accuracy (0.691) and highest F1 score (0.676). In addition, it also has strong recall and precision. Overall, Model 6 provides the strongest performance. Although Models 4 and 5 performed similarly, Model 4 has the same accuracy and F1 score as Model 6, but a lower ROC-AUC; Model 5 has almost the same performance, but Model 6 is still slightly better in both accuracy and ROC-AUC.